# Discover Alternative Rank-23 Schemes & Test for Rank-22 Reducibility

**Goal**: Generate distinct rank-23 factorizations of the 3x3 MM tensor (not just Laderman's),
then test each for reducibility to rank-22.

**Strategy**:
- Phase 1: Random orthogonal basis + random init to find different rank-23 local minima
- Phase 2: For each discovered scheme, drop each column and optimize for rank-22

**Reference**: Research analysis suggests 17,000+ distinct rank-23 schemes may exist;
some may NOT have the "orthogonal anchor" barrier that makes rank-22 impossible from Laderman's.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import time, os, json
from collections import defaultdict, Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Load the known rank-23 factorization (for reference / ground truth)
def load_data(path="factorizations_r.npz"):
    d = np.load(path, allow_pickle=True)
    t = d["3,3,3"]
    if t.ndim == 0: t = t.item()
    if isinstance(t,dict): U,V,W = t["u"],t["v"],t["w"]
    else: U,V,W = t
    return np.array(U,dtype=np.float32), np.array(V,dtype=np.float32), np.array(W,dtype=np.float32)

U_ref, V_ref, W_ref = load_data()
print(f"Reference rank-23: U {U_ref.shape}")

def build_target():
    T = torch.zeros((9,9,9), dtype=torch.float32, device=DEVICE)
    for i in range(3):
        for j in range(3):
            for k in range(3):
                T[i*3+j, j*3+k, i*3+k] = 1.0
    return T

T_target = build_target()
print(f"Target tensor: {T_target.shape}")

# Verify reference is correct
U_t = torch.tensor(U_ref, device=DEVICE)
V_t = torch.tensor(V_ref, device=DEVICE)
W_t = torch.tensor(W_ref, device=DEVICE)
ref_mse = F.mse_loss(torch.einsum("ir,jr,kr->ijk",U_t,V_t,W_t), T_target).item()
print(f"Reference MSE: {ref_mse:.2e}")

## Phase 1: Discover Alternative Rank-23 Schemes

Key difference from v1 notebook:
- Initialize from **random** (not from the known factorization)
- Use basis randomization to explore different decomposition landscapes
- Collect all distinct rank-23 schemes that converge

In [ ]:
class Config:
    N_SCHEMES = 20  # Number of alternative schemes to discover
    EPOCHS = 15000  # Per scheme
    LR = 0.005
    NOISE_INIT = 0.1
    PHASES = [
        (0, 3000, 0.10, 0.005, 0.0),
        (3000, 8000, 0.04, 0.002, 1e-6),
        (8000, 12000, 0.01, 0.001, 1e-5),
        (12000, 15000, 0.0, 0.0005, 1e-5),
    ]
    VERBOSE_INTERVAL = 2000
    CONVERGE_THRESHOLD = 1e-6  # MSE below this = exact factorization found
    PLATEAU_PATIENCE = 1000
    PLATEAU_BURST = 0.03

cfg = Config()

def random_orthogonal(dim):
    H = torch.randn(dim, dim, device=DEVICE)
    Q, R = torch.linalg.qr(H)
    return Q * torch.sign(torch.diag(R)).unsqueeze(0)

def apply_basis(T, P, Q, R):
    T1 = torch.einsum("ia,abc->ibc", P, T)
    T2 = torch.einsum("jb,ibc->ijc", Q, T1)
    T3 = torch.einsum("kc,ijc->ijk", R, T2)
    return T3

def get_params(epoch, phases):
    for s,e,n,lr,l1 in phases:
        if s <= epoch < e:
            frac = (epoch-s)/(e-s) if e>s else 0.0
            return n*(1-0.5*frac), lr, l1
    return 0.0, 1e-4, phases[-1][4]

def discover_scheme(tid, verbose=True):
    """Discover a rank-23 factorization via basis randomization + random init."""
    start_t = time.time()
    
    # Random orthogonal basis
    P, Q, R = random_orthogonal(9), random_orthogonal(9), random_orthogonal(9)
    T_trans = apply_basis(T_target, P, Q, R)
    
    # Random initialization (NOT from reference)
    U = torch.randn(9, 23, device=DEVICE) * cfg.NOISE_INIT
    V = torch.randn(9, 23, device=DEVICE) * cfg.NOISE_INIT
    W = torch.randn(9, 23, device=DEVICE) * cfg.NOISE_INIT
    U.requires_grad_(True)
    V.requires_grad_(True)
    W.requires_grad_(True)
    
    opt = torch.optim.Adam([U, V, W], lr=cfg.LR)
    best_loss = float("inf")
    best_state = None
    patience = 0
    
    for ep in range(cfg.EPOCHS):
        nmag, lr_curr, l1s = get_params(ep, cfg.PHASES)
        for g in opt.param_groups:
            g["lr"] = lr_curr
        opt.zero_grad()
        
        recon = torch.einsum("ir,jr,kr->ijk", U, V, W)
        mse = F.mse_loss(recon, T_trans)
        reg = l1s * (U.abs().mean() + V.abs().mean() + W.abs().mean())
        (mse + reg).backward()
        opt.step()
        
        if nmag > 0:
            with torch.no_grad():
                U.add_(torch.randn_like(U) * nmag)
                V.add_(torch.randn_like(V) * nmag)
                W.add_(torch.randn_like(W) * nmag)
        
        cv = mse.item()
        if cv < best_loss:
            best_loss = cv
            best_state = (U.detach().clone(), V.detach().clone(), W.detach().clone(), P, Q, R)
            patience = 0
        else:
            patience += 1
        
        if patience > cfg.PLATEAU_PATIENCE and ep > cfg.PHASES[1][0]:
            with torch.no_grad():
                U.add_(torch.randn_like(U) * cfg.PLATEAU_BURST)
                V.add_(torch.randn_like(V) * cfg.PLATEAU_BURST)
                W.add_(torch.randn_like(W) * cfg.PLATEAU_BURST)
            patience = 0
        
        if verbose and ep % cfg.VERBOSE_INTERVAL == 0:
            print(f"  S{tid:2d} Ep{ep:5d} MSE:{cv:.2e}")
        
        if cv < cfg.CONVERGE_THRESHOLD:
            if verbose:
                print(f"  S{tid:2d}: EXACT at ep{ep}!")
            break
    
    U_b, V_b, W_b, Pb, Qb, Rb = best_state
    with torch.no_grad():
        # Transform back to original basis
        U_o = Pb.T @ U_b
        V_o = Qb.T @ V_b
        W_o = Rb.T @ W_b
        mse_o = F.mse_loss(
            torch.einsum("ir,jr,kr->ijk", U_o, V_o, W_o), T_target
        ).item()
    
    return {
        "tid": tid,
        "mse": mse_o,
        "mse_t": best_loss,
        "epoch": ep + 1,
        "time": time.time() - start_t,
        "U": U_o.cpu(),
        "V": V_o.cpu(),
        "W": W_o.cpu(),
    }

print("Phase 1 ready: discover_scheme()")

In [ ]:
# Run Phase 1: Discover N alternative rank-23 schemes
print(f"\n{'='*60}")
print(f"Phase 1: Discovering {cfg.N_SCHEMES} alternative rank-23 schemes")
print(f"{'='*60}\n")

schemes = []
t0 = time.time()

for s in range(cfg.N_SCHEMES):
    print(f"\n--- Scheme {s+1}/{cfg.N_SCHEMES} ---")
    r = discover_scheme(s + 1, verbose=True)
    tag = " *** BEST ***" if len(schemes) == 0 or r["mse"] < min(x["mse"] for x in schemes) else ""
    print(f"  >>> MSE: {r['mse']:.2e} | {r['time']:.0f}s{tag}")
    schemes.append(r)

t1 = time.time()
print(f"\nPhase 1 complete: {t1-t0:.0f}s")
print(f"Schemes found: {len(schemes)}")

# Show MSE distribution
mses = [s["mse"] for s in schemes]
print(f"MSE range: {min(mses):.2e} to {max(mses):.2e}")
print(f"MSE median: {sorted(mses)[len(mses)//2]:.2e}")

In [ ]:
# Filter to well-converged schemes (MSE < 1e-4 means essentially exact rank-23)
good_schemes = [s for s in schemes if s["mse"] < 1e-4]
print(f"Well-converged schemes (MSE < 1e-4): {len(good_schemes)}/{len(schemes)}")

# If we found schemes, compare them to the reference to check they're genuinely different
if len(good_schemes) > 0:
    U_ref_t = torch.tensor(U_ref)
    V_ref_t = torch.tensor(V_ref)
    W_ref_t = torch.tensor(W_ref)
    
    for s in good_schemes[:5]:
        # Compute alignment: the Frobenius norm difference
        U_diff = torch.norm(s["U"] - U_ref_t).item()
        print(f"  Scheme {s['tid']}: ||U - U_ref|| = {U_diff:.2f}  |  MSE = {s['mse']:.2e}")
else:
    print("No well-converged schemes found. Check optimization.")
    # Use all schemes with best MSE
    good_schemes = schemes

## Phase 2: Test Each Scheme for Rank-22 Reducibility

For each discovered scheme, try dropping each column and optimizing.
If any column-drop achieves MSE significantly below the ~7e-3 barrier, that scheme is promising.

In [ ]:
class TestConfig:
    EPOCHS_PER_DROP = 15000
    PHASES = [
        (0, 3000, 0.12, 0.005, 0.0),
        (3000, 8000, 0.06, 0.002, 1e-5),
        (8000, 13000, 0.02, 0.001, 1e-4),
        (13000, 15000, 0.0, 0.0005, 0.0),
    ]
    NOISE_INIT = 0.05
    PLATEAU_PATIENCE = 800
    PLATEAU_BURST = 0.04
    VERBOSE_INTERVAL = 3000
    # The barrier from Laderman's: ~7e-3.
    # If ANY scheme+column achieves << 1e-3, we have a breakthrough.
    BREAKTHROUGH_THRESHOLD = 5e-4

tcfg = TestConfig()

def test_rank22(scheme_id, U_full, V_full, W_full, col_drop, verbose=True):
    """Drop one column from a rank-23 scheme and optimize for rank-22."""
    start_t = time.time()
    
    keep = [i for i in range(23) if i != col_drop]
    
    # Initialize from the scheme (with small noise)
    U_init = U_full[:, keep].float().clone().to(DEVICE)
    V_init = V_full[:, keep].float().clone().to(DEVICE)
    W_init = W_full[:, keep].float().clone().to(DEVICE)
    
    U = U_init + tcfg.NOISE_INIT * torch.randn(9, 22, device=DEVICE)
    V = V_init + tcfg.NOISE_INIT * torch.randn(9, 22, device=DEVICE)
    W = W_init + tcfg.NOISE_INIT * torch.randn(9, 22, device=DEVICE)
    U.requires_grad_(True)
    V.requires_grad_(True)
    W.requires_grad_(True)
    
    opt = torch.optim.Adam([U, V, W], lr=0.005)
    best_loss = float("inf")
    patience = 0
    
    for ep in range(tcfg.EPOCHS_PER_DROP):
        nmag, lr_curr, l1s = get_params(ep, tcfg.PHASES)
        for g in opt.param_groups:
            g["lr"] = lr_curr
        opt.zero_grad()
        
        mse = F.mse_loss(torch.einsum("ir,jr,kr->ijk", U, V, W), T_target)
        reg = l1s * (U.abs().mean() + V.abs().mean() + W.abs().mean())
        (mse + reg).backward()
        opt.step()
        
        if nmag > 0:
            with torch.no_grad():
                U.add_(torch.randn_like(U) * nmag)
                V.add_(torch.randn_like(V) * nmag)
                W.add_(torch.randn_like(W) * nmag)
        
        cv = mse.item()
        if cv < best_loss:
            best_loss = cv
            patience = 0
            if cv < 1e-8:
                if verbose:
                    print(f"  >>> EXACT RANK-22! col={col_drop} ep={ep}")
                break
        else:
            patience += 1
        
        if patience > tcfg.PLATEAU_PATIENCE and ep > tcfg.PHASES[1][0]:
            with torch.no_grad():
                U.add_(torch.randn_like(U) * tcfg.PLATEAU_BURST)
                V.add_(torch.randn_like(V) * tcfg.PLATEAU_BURST)
                W.add_(torch.randn_like(W) * tcfg.PLATEAU_BURST)
            patience = 0
        
        if verbose and ep % tcfg.VERBOSE_INTERVAL == 0:
            print(f"    S{scheme_id} C{col_drop} Ep{ep:5d} MSE:{cv:.2e}")
    
    return {
        "scheme": scheme_id,
        "col": col_drop,
        "mse": best_loss,
        "epoch": ep + 1,
        "time": time.time() - start_t,
    }

print("Phase 2 ready: test_rank22()")

In [ ]:
# Run Phase 2 on the best (most different) schemes
# Limit to top 3 schemes to keep runtime manageable (3 x 23 = 69 trials)
# Increase this if you have more compute time

MAX_SCHEMES_TO_TEST = 3

# Pick schemes that are both well-converged AND different from reference
candidates = sorted(good_schemes, key=lambda s: s["mse"])[:MAX_SCHEMES_TO_TEST]

print(f"\n{'='*60}")
print(f"Phase 2: Testing {len(candidates)} schemes x 23 column drops")
print(f"{'='*60}\n")

all_results = []
breakthroughs = []
baseline_barrier = 0.007  # Laderman's known barrier

for scheme in candidates:
    sid = scheme["tid"]
    print(f"\n--- Testing Scheme {sid} (MSE={scheme['mse']:.2e}) ---")
    
    scheme_best = float("inf")
    scheme_worst = 0.0
    
    for col in range(23):
        r = test_rank22(sid, scheme["U"], scheme["V"], scheme["W"], col, verbose=False)
        all_results.append(r)
        
        scheme_best = min(scheme_best, r["mse"])
        scheme_worst = max(scheme_worst, r["mse"])
        
        tag = ""
        if r["mse"] < tcfg.BREAKTHROUGH_THRESHOLD:
            tag = " *** BREAKTHROUGH! ***"
            breakthroughs.append(r)
        elif r["mse"] < baseline_barrier:
            tag = " *** BELOW LADERMAN BARRIER! ***"
        
        print(f"  Col {col:2d}: MSE={r['mse']:.2e} | {r['time']:.0f}s{tag}")
    
    print(f"  >> Scheme {sid}: best={scheme_best:.2e} worst={scheme_worst:.2e}")

In [ ]:
# === Summary ===
print(f"\n{'='*60}")
print("RESULTS SUMMARY")
print(f"{'='*60}\n")

if breakthroughs:
    print(f"*** BREAKTHROUGHS FOUND: {len(breakthroughs)} ***")
    for r in breakthroughs:
        print(f"  Scheme {r['scheme']}, Col {r['col']}: MSE={r['mse']:.2e}")
else:
    print("No breakthroughs (MSE < 5e-4)")

# Best result across all schemes
if all_results:
    best = min(all_results, key=lambda r: r["mse"])
    print(f"\nBest overall: Scheme {best['scheme']}, Col {best['col']}: MSE={best['mse']:.2e}")
    
    # How many below Laderman barrier?
    below_barrier = [r for r in all_results if r["mse"] < baseline_barrier]
    print(f"Below Laderman barrier (~7e-3): {len(below_barrier)}/{len(all_results)}")
    
    # Per-scheme summary
    by_scheme = defaultdict(list)
    for r in all_results:
        by_scheme[r["scheme"]].append(r["mse"])
    
    print(f"\nPer-scheme stats:")
    for sid, ms in sorted(by_scheme.items()):
        print(f"  Scheme {sid}: min={min(ms):.2e} median={sorted(ms)[len(ms)//2]:.2e} max={max(ms):.2e}")
    
    # Compare with reference (Laderman's scheme)
    print(f"\nReference (Laderman) v1 barrier: ~7.22e-03 to ~8e-03")

In [ ]:
# Save all results
out_dir = "outputs"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

import pickle
with open(os.path.join(out_dir, "scheme_discovery_results.pkl"), "wb") as f:
    pickle.dump({
        "schemes": [
            {"tid": s["tid"], "mse": s["mse"]}
            for s in schemes
        ],
        "rank22_results": all_results,
        "breakthroughs": breakthroughs,
        "config": {
            "n_schemes": cfg.N_SCHEMES,
            "epochs_per_scheme": cfg.EPOCHS,
            "epochs_per_drop": tcfg.EPOCHS_PER_DROP,
        },
    }, f)

print(f"Results saved to {out_dir}/scheme_discovery_results.pkl")

# Save best factorization if found
if breakthroughs:
    best_breakthrough = min(breakthroughs, key=lambda r: r["mse"])
    # Find the corresponding scheme
    for s in schemes:
        if s["tid"] == best_breakthrough["scheme"]:
            keep = [i for i in range(23) if i != best_breakthrough["col"]]
            U_r22 = s["U"][:, keep].numpy()
            V_r22 = s["V"][:, keep].numpy()
            W_r22 = s["W"][:, keep].numpy()
            np.savez(
                os.path.join(out_dir, "r22_breakthrough.npz"),
                U=U_r22, V=V_r22, W=W_r22,
            )
            print(f"Breakthrough factorization saved to {out_dir}/r22_breakthrough.npz")
            break

In [ ]:
# Optional: Fine-tune any promising result with continuous optimization
# If we found something below the barrier, this might push it to zero

def fine_tune_continuous(Ui, Vi, Wi, n_epochs=10000):
    U = Ui.float().clone().detach().requires_grad_(True)
    V = Vi.float().clone().detach().requires_grad_(True)
    W = Wi.float().clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([U, V, W], lr=0.001)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    best = float("inf")
    best_s = None
    for ep in range(n_epochs):
        opt.zero_grad()
        mse = F.mse_loss(torch.einsum("ir,jr,kr->ijk", U, V, W), T_target)
        l1 = 1e-6 * (U.abs().mean() + V.abs().mean() + W.abs().mean())
        (mse + l1).backward()
        opt.step()
        sch.step()
        cv = mse.item()
        if cv < best:
            best = cv
            best_s = (U.detach().clone(), V.detach().clone(), W.detach().clone())
        if best < 1e-10:
            break
    Uf, Vf, Wf = best_s
    return Uf, Vf, Wf, best

if breakthroughs:
    best_r = min(breakthroughs, key=lambda r: r["mse"])
    print(f"Fine-tuning breakthrough: Scheme {best_r['scheme']}, Col {best_r['col']}")
    
    for s in schemes:
        if s["tid"] == best_r["scheme"]:
            keep = [i for i in range(23) if i != best_r["col"]]
            Ui = s["U"][:, keep].float().to(DEVICE)
            Vi = s["V"][:, keep].float().to(DEVICE)
            Wi = s["W"][:, keep].float().to(DEVICE)
            
            Uf, Vf, Wf, mse_ft = fine_tune_continuous(Ui, Vi, Wi)
            print(f"After fine-tune: MSE = {mse_ft:.2e}")
            
            if mse_ft < 1e-6:
                print("*** EXACT RANK-22 FOUND IN CONTINUOUS! ***")
            break
else:
    # Try fine-tuning the best result anyway
    if all_results:
        best_r = min(all_results, key=lambda r: r["mse"])
        print(f"No breakthroughs, but fine-tuning best: Scheme {best_r['scheme']}, Col {best_r['col']} (MSE={best_r['mse']:.2e})")